# Concept Mapping Deep Dive

This notebook explores the concept mapping capabilities of the Portiere SDK in depth.
Concept mapping is the process of translating local hospital codes (ICD-10, local drug codes,
internal lab identifiers) into standardized vocabularies used by the OMOP Common Data Model.

## What is Concept Mapping?

Healthcare institutions use a variety of coding systems to represent clinical data:

- **ICD-10-CM** -- Diagnosis codes used for billing and clinical documentation
- **NDC** -- National Drug Codes for medications
- **Local codes** -- Institution-specific identifiers for labs, procedures, etc.

The OMOP CDM requires all clinical data to be expressed using **standard concepts** from
controlled vocabularies:

| Source Domain | Target Vocabulary | Example |
|---|---|---|
| Diagnoses (ICD-10) | SNOMED CT | E11.9 -> 201826 (Type 2 diabetes mellitus) |
| Lab tests | LOINC | Local "BUN" -> 3094 (Urea nitrogen) |
| Medications | RxNorm | NDC 00002-1433 -> 1361568 (Metformin) |
| Procedures (CPT) | SNOMED CT | 99213 -> Office visit concept |

Portiere automates this mapping using hybrid search (dense embeddings + BM25) with
confidence-based routing:

- **>= 0.95**: Auto-accepted (high confidence)
- **0.80 - 0.95**: Needs verification (review recommended)
- **0.70 - 0.80**: Needs manual review
- **< 0.70**: Manual mapping required

**Embedding providers:** By default, Portiere uses HuggingFace SapBERT for biomedical
embeddings. You can switch to Ollama, OpenAI, or Portiere Cloud via `EmbeddingConfig`.
See [04_knowledge_backends](./04_knowledge_backends.ipynb) for provider examples.

## 1. Project Setup

import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig
from portiere.engines import PolarsEngine

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
)

project = portiere.init(name="Concept Mapping Demo", engine=PolarsEngine(), config=config)
print(f"Project initialized: {project.name}")

In [1]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


In [2]:
import portiere
from portiere.engines import PolarsEngine

project = portiere.init(name="Concept Mapping Demo", engine=PolarsEngine())
print(f"Project initialized: {project.name}")

2026-04-16 00:26:57 [info     ] PolarsEngine initialized      


2026-04-16 00:26:57 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project initialized: Concept Mapping Demo


## 2. Mapping ICD-10 Diagnosis Codes

We start with a set of common ICD-10-CM diagnosis codes. Each code includes:
- `code`: The ICD-10-CM code string
- `description`: Human-readable description of the diagnosis
- `count`: How many times this code appears in the source data (helps prioritize review)

In [3]:
diagnosis_codes = [
    {"code": "E11.9", "description": "Type 2 diabetes mellitus, without complications", "count": 1523},
    {"code": "I10", "description": "Essential (primary) hypertension", "count": 2891},
    {"code": "J18.9", "description": "Pneumonia, unspecified organism", "count": 456},
    {"code": "N18.6", "description": "End stage renal disease", "count": 234},
    {"code": "Z87.891", "description": "Personal history of nicotine dependence", "count": 1102},
    {"code": "F32.9", "description": "Major depressive disorder, single episode, unspecified", "count": 891},
]

concept_map = project.map_concepts(
    codes=diagnosis_codes,
    vocabularies=["SNOMED", "ICD10CM"],
)

print(concept_map)

2026-04-16 00:26:57 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_knowledge_layer message='No knowledge_layer configured. Concept search will not work.'


2026-04-16 00:26:57 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=None llm_verifier=False reranker=True


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Type 2 diabetes mellitus, without complications'


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Essential (primary) hypertension'


...

2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Major depressive disorder, single episode, unspeci'


2026-04-16 00:26:57 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=6


2026-04-16 00:26:57 [info     ] local_storage.concept_mapping_saved items_count=6 project='Concept Mapping Demo'


2026-04-16 00:26:57 [info     ] project.concepts_mapped        auto_rate=0.0 project='Concept Mapping Demo' total=6


items=[ConceptMappingItem(source_code='E11.9', source_description='Type 2 diabetes mellitus, without complications', source_column=None, source_count=1523, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappingItem(source_code='I10', source_description='Essential (primary) hypertension', source_column=None, source_count=2891, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappingItem(source_code='J18.9', source_description='Pneumonia, unspecified organism', source_column=None, source_count=456, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappin

## 3. Understanding Mapping Results

The `summary()` method provides an overview of how the mapping performed across all codes.

In [4]:
summary = concept_map.summary()

print("Concept Mapping Summary")
print("=" * 40)
print(f"Total codes:      {summary['total']}")
print(f"Auto-mapped:      {summary['auto_mapped']}")
print(f"Needs review:     {summary['needs_review']}")
print(f"Manual required:  {summary['manual_required']}")
print(f"Auto-accept rate: {summary['auto_rate']:.1%}")
print(f"Coverage:         {summary['coverage']:.1%}")

Concept Mapping Summary
Total codes:      6
Auto-mapped:      0
Needs review:     0
Manual required:  6
Auto-accept rate: 0.0%
Coverage:         0.0%


## 4. Inspecting Candidates for Each Mapping

Each mapping item contains a ranked list of candidate concepts. The top candidate is
automatically selected as the best match, but alternative candidates are available
for review. Each candidate includes:

- `concept_id`: OMOP concept identifier
- `concept_name`: Human-readable concept name
- `vocabulary_id`: Source vocabulary (e.g., SNOMED, ICD10CM)
- `domain_id`: Clinical domain (Condition, Drug, Measurement, etc.)
- `score`: Similarity score from the hybrid search engine

In [5]:
for item in concept_map.items:
    print(f"\n{item.source_code}: {item.source_description}")
    print(f"  Best match: {item.target_concept_name} (concept_id={item.target_concept_id})")
    print(f"  Confidence: {item.confidence:.2f}, Method: {item.method.value}")
    if item.candidates:
        print(f"  All candidates:")
        for i, c in enumerate(item.candidates[:5]):
            print(f"    [{i}] {c.concept_name} ({c.vocabulary_id}, score={c.score:.3f})")


E11.9: Type 2 diabetes mellitus, without complications
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual

I10: Essential (primary) hypertension
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual

J18.9: Pneumonia, unspecified organism
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual

N18.6: End stage renal disease
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual

Z87.891: Personal history of nicotine dependence
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual

F32.9: Major depressive disorder, single episode, unspecified
  Best match:  (concept_id=None)
  Confidence: 0.00, Method: manual


## 5. Filtering by Mapping Status

Portiere provides convenience methods to filter items by their current status.
This is useful for building review workflows where you process items in batches.

In [6]:
# Items that were auto-accepted (confidence >= threshold)
auto_items = concept_map.auto_mapped()
print(f"Auto-mapped ({len(auto_items)} items):")
for item in auto_items:
    print(f"  {item.source_code} -> {item.target_concept_name} ({item.confidence:.2f})")

print()

# Items that need human review
review_items = concept_map.needs_review()
print(f"Needs review ({len(review_items)} items):")
for item in review_items:
    print(f"  {item.source_code} -> {item.target_concept_name} ({item.confidence:.2f})")

print()

# Unmapped items (below threshold or no match found)
unmapped_items = concept_map.unmapped()
print(f"Unmapped ({len(unmapped_items)} items):")
for item in unmapped_items:
    print(f"  {item.source_code}: {item.source_description}")

Auto-mapped (0 items):

Needs review (0 items):

Unmapped (6 items):
  E11.9: Type 2 diabetes mellitus, without complications
  I10: Essential (primary) hypertension
  J18.9: Pneumonia, unspecified organism
  N18.6: End stage renal disease
  Z87.891: Personal history of nicotine dependence
  F32.9: Major depressive disorder, single episode, unspecified


## 6. Review Workflow -- Approve, Reject, Override

The SDK provides three actions for reviewing individual mapping items:

- **`approve(candidate_index=N)`**: Accept a candidate. Index 0 uses the top match (AUTO method).
  Using any other index sets the method to OVERRIDE.
- **`reject()`**: Mark the item as UNMAPPED, indicating no suitable concept exists.
- **`override(concept_id, concept_name, vocabulary_id)`**: Manually specify the target concept
  when none of the candidates are correct.

In [7]:
# Example: Approve the top candidate for the first review item
review_items = concept_map.needs_review()
if review_items:
    item = review_items[0]
    print(f"Reviewing: {item.source_code} - {item.source_description}")
    print(f"  Current match: {item.target_concept_name} (confidence={item.confidence:.2f})")
    print(f"  Candidates:")
    for i, c in enumerate(item.candidates[:3]):
        print(f"    [{i}] {c.concept_name} ({c.vocabulary_id}, score={c.score:.3f})")

    # Approve with the top candidate
    item.approve(candidate_index=0)
    print(f"\n  -> Approved: {item.target_concept_name} (method={item.method.value})")

In [8]:
# Example: Override with a specific concept when candidates are not ideal
review_items = concept_map.needs_review()
if review_items:
    item = review_items[0]
    print(f"Reviewing: {item.source_code} - {item.source_description}")
    print(f"  Current match: {item.target_concept_name}")

    # Override with a manually specified concept
    item.override(
        concept_id=443614,
        concept_name="Type 2 diabetes mellitus",
        vocabulary_id="SNOMED",
    )
    print(f"  -> Overridden to: {item.target_concept_name} (method={item.method.value})")

In [9]:
# Example: Reject an item (mark as UNMAPPED)
review_items = concept_map.needs_review()
if review_items:
    item = review_items[0]
    print(f"Rejecting: {item.source_code} - {item.source_description}")
    item.reject()
    print(f"  -> Status: {item.method.value}")

In [10]:
# Approve the second-best candidate (sets method to OVERRIDE)
review_items = concept_map.needs_review()
if review_items:
    item = review_items[0]
    print(f"Reviewing: {item.source_code} - {item.source_description}")
    if len(item.candidates) >= 2:
        print(f"  Selecting candidate [1]: {item.candidates[1].concept_name}")
        item.approve(candidate_index=1)
        print(f"  -> Approved: {item.target_concept_name} (method={item.method.value})")

## 7. Bulk Actions

For large mapping sets, you can approve all items that are in review status
with their top candidate, then finalize the entire mapping.

In [11]:
# Approve all remaining review items with their top candidate
concept_map.approve_all()

# Check the updated summary
summary = concept_map.summary()
print(f"After approve_all:")
print(f"  Auto-mapped:     {summary['auto_mapped']}")
print(f"  Needs review:    {summary['needs_review']}")
print(f"  Manual required: {summary['manual_required']}")
print(f"  Coverage:        {summary['coverage']:.1%}")

After approve_all:
  Auto-mapped:     0
  Needs review:    0
  Manual required: 6
  Coverage:        0.0%


In [12]:
# Finalize the mapping -- locks all items and prepares for ETL
concept_map.finalize()
print("Concept mapping finalized.")

Concept mapping finalized.


## 8. Export to OMOP source_to_concept_map

The `to_source_to_concept_map()` method exports the finalized mapping in the
OMOP CDM `source_to_concept_map` table format. This is the standard format
for documenting how source codes were mapped to OMOP concepts.

In [13]:
stcm = concept_map.to_source_to_concept_map()

import pandas as pd

df = pd.DataFrame(stcm)
print(df.to_string(index=False))

Empty DataFrame
Columns: []
Index: []


In [14]:
# Save the source_to_concept_map to CSV for downstream use
df.to_csv("source_to_concept_map.csv", index=False)
print(f"Exported {len(df)} mappings to source_to_concept_map.csv")
print(f"\nColumns: {list(df.columns)}")

Exported 0 mappings to source_to_concept_map.csv

Columns: []


## 9. Handling Unmapped Codes

Some source codes may not have a suitable standard concept. These need special attention.
Common strategies for handling unmapped codes include:

1. **Search broader vocabularies** -- Try additional target vocabularies
2. **Use parent/ancestor codes** -- Map to a less specific but valid concept
3. **Custom concepts** -- Create institution-specific concepts (concept_id >= 2000000000)
4. **Map to concept_id = 0** -- The OMOP convention for "no matching concept"

In [15]:
# Re-run mapping with some codes that are harder to map
tricky_codes = [
    {"code": "R69", "description": "Illness, unspecified", "count": 45},
    {"code": "Z99.2", "description": "Dependence on renal dialysis", "count": 112},
    {"code": "LOCAL-001", "description": "Internal screening code", "count": 500},
]

tricky_map = project.map_concepts(
    codes=tricky_codes,
    vocabularies=["SNOMED", "ICD10CM"],
)

# Check for unmapped items
unmapped = tricky_map.unmapped()
print(f"Unmapped codes: {len(unmapped)}")
for item in unmapped:
    print(f"  {item.source_code}: {item.source_description}")
    print(f"    Confidence: {item.confidence:.2f}")
    if item.candidates:
        print(f"    Best candidate: {item.candidates[0].concept_name} (score={item.candidates[0].score:.3f})")
    else:
        print(f"    No candidates found")

2026-04-16 00:26:57 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_knowledge_layer message='No knowledge_layer configured. Concept search will not work.'


2026-04-16 00:26:57 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=None llm_verifier=False reranker=True


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Illness, unspecified'


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Dependence on renal dialysis'


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Internal screening code'


2026-04-16 00:26:57 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=3


2026-04-16 00:26:57 [info     ] local_storage.concept_mapping_saved items_count=3 project='Concept Mapping Demo'


2026-04-16 00:26:57 [info     ] project.concepts_mapped        auto_rate=0.0 project='Concept Mapping Demo' total=3


Unmapped codes: 3
  R69: Illness, unspecified
    Confidence: 0.00
    No candidates found
  Z99.2: Dependence on renal dialysis
    Confidence: 0.00
    No candidates found
  LOCAL-001: Internal screening code
    Confidence: 0.00
    No candidates found


In [16]:
# Strategy: Override unmapped items with manually researched concepts
for item in tricky_map.unmapped():
    if item.source_code == "LOCAL-001":
        # Local codes with no standard equivalent map to concept_id 0
        item.override(
            concept_id=0,
            concept_name="No matching concept",
            vocabulary_id="None",
        )
        print(f"  {item.source_code} -> concept_id=0 (no match)")
    elif item.candidates:
        # Accept the best available candidate even if below threshold
        item.approve(candidate_index=0)
        print(f"  {item.source_code} -> {item.target_concept_name} (manually accepted)")

  LOCAL-001 -> concept_id=0 (no match)


## 10. Mapping Lab Codes (LOINC)

Laboratory test codes are mapped to LOINC (Logical Observation Identifiers Names and Codes),
the standard vocabulary for measurements in OMOP.

In [17]:
lab_codes = [
    {"code": "GLU", "description": "Glucose, serum", "count": 5200},
    {"code": "HBA1C", "description": "Hemoglobin A1c", "count": 3100},
    {"code": "BUN", "description": "Blood urea nitrogen", "count": 4800},
    {"code": "CREAT", "description": "Creatinine, serum", "count": 4750},
    {"code": "WBC", "description": "White blood cell count", "count": 6100},
    {"code": "PLT", "description": "Platelet count", "count": 5900},
]

lab_map = project.map_concepts(
    codes=lab_codes,
    vocabularies=["LOINC"],
)

print("Lab Code Mappings:")
print("-" * 80)
for item in lab_map.items:
    print(f"  {item.source_code:8s} -> {item.target_concept_name or 'UNMAPPED':40s} "
          f"(confidence={item.confidence:.2f}, method={item.method.value})")

print(f"\nLab mapping summary:")
lab_summary = lab_map.summary()
print(f"  Auto-mapped: {lab_summary['auto_mapped']}/{lab_summary['total']}")
print(f"  Coverage:    {lab_summary['coverage']:.1%}")

2026-04-16 00:26:57 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_knowledge_layer message='No knowledge_layer configured. Concept search will not work.'


2026-04-16 00:26:57 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=None llm_verifier=False reranker=True


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Glucose, serum'


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Hemoglobin A1c'


...

2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Platelet count'


2026-04-16 00:26:57 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=6


2026-04-16 00:26:57 [info     ] local_storage.concept_mapping_saved items_count=6 project='Concept Mapping Demo'


2026-04-16 00:26:57 [info     ] project.concepts_mapped        auto_rate=0.0 project='Concept Mapping Demo' total=6


Lab Code Mappings:
--------------------------------------------------------------------------------
  GLU      -> UNMAPPED                                 (confidence=0.00, method=manual)
  HBA1C    -> UNMAPPED                                 (confidence=0.00, method=manual)
  BUN      -> UNMAPPED                                 (confidence=0.00, method=manual)
  CREAT    -> UNMAPPED                                 (confidence=0.00, method=manual)
  WBC      -> UNMAPPED                                 (confidence=0.00, method=manual)
  PLT      -> UNMAPPED                                 (confidence=0.00, method=manual)

Lab mapping summary:
  Auto-mapped: 0/6
  Coverage:    0.0%


## 11. Mapping Drug Codes (RxNorm)

Medication codes are mapped to RxNorm, the standard vocabulary for drugs in OMOP.
Source codes might be NDC codes, local formulary codes, or drug name strings.

In [18]:
drug_codes = [
    {"code": "MET500", "description": "Metformin 500mg oral tablet", "count": 2300},
    {"code": "LISIN10", "description": "Lisinopril 10mg oral tablet", "count": 1850},
    {"code": "ATOR20", "description": "Atorvastatin 20mg oral tablet", "count": 1600},
    {"code": "AMOX500", "description": "Amoxicillin 500mg oral capsule", "count": 900},
    {"code": "OMEP20", "description": "Omeprazole 20mg oral capsule", "count": 1100},
]

drug_map = project.map_concepts(
    codes=drug_codes,
    vocabularies=["RxNorm"],
)

print("Drug Code Mappings:")
print("-" * 90)
for item in drug_map.items:
    print(f"  {item.source_code:10s} -> {item.target_concept_name or 'UNMAPPED':45s} "
          f"(confidence={item.confidence:.2f})")

drug_summary = drug_map.summary()
print(f"\nDrug mapping: {drug_summary['auto_mapped']}/{drug_summary['total']} auto-mapped, "
      f"coverage={drug_summary['coverage']:.1%}")

2026-04-16 00:26:57 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_knowledge_layer message='No knowledge_layer configured. Concept search will not work.'


2026-04-16 00:26:57 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=None llm_verifier=False reranker=True


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Metformin 500mg oral tablet'


2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Lisinopril 10mg oral tablet'


...

2026-04-16 00:26:57 [warning  ] local_concept_mapper.no_backend query='Omeprazole 20mg oral capsule'


2026-04-16 00:26:57 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=5


2026-04-16 00:26:57 [info     ] local_storage.concept_mapping_saved items_count=5 project='Concept Mapping Demo'


2026-04-16 00:26:57 [info     ] project.concepts_mapped        auto_rate=0.0 project='Concept Mapping Demo' total=5


Drug Code Mappings:
------------------------------------------------------------------------------------------
  MET500     -> UNMAPPED                                      (confidence=0.00)
  LISIN10    -> UNMAPPED                                      (confidence=0.00)
  ATOR20     -> UNMAPPED                                      (confidence=0.00)
  AMOX500    -> UNMAPPED                                      (confidence=0.00)
  OMEP20     -> UNMAPPED                                      (confidence=0.00)

Drug mapping: 0/5 auto-mapped, coverage=0.0%


## 12. Mapping from Source File Columns

Instead of providing codes manually, you can map concepts directly from columns
in a source file. Portiere will extract unique codes from the specified columns
and map them in bulk.

In [19]:
# Add a source file and map concepts from its columns
source = project.add_source("example_assets/07_concept_mapping_deep_dive/diagnoses.csv")

file_concept_map = project.map_concepts(
    source=source,
    code_columns=["diagnosis_code"]  # diagnoses.csv has no drug_code column,
)

file_summary = file_concept_map.summary()
print(f"Source file concept mapping:")
print(f"  Total unique codes: {file_summary['total']}")
print(f"  Auto-mapped:        {file_summary['auto_mapped']}")
print(f"  Needs review:       {file_summary['needs_review']}")
print(f"  Coverage:           {file_summary['coverage']:.1%}")

2026-04-16 00:26:57 [info     ] project.source_added           project='Concept Mapping Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:27:00 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:27:00 [info     ] project.profiled               source=diagnoses


2026-04-16 00:27:00 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=None


...

2026-04-16 00:27:00 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=8


2026-04-16 00:27:00 [info     ] local_storage.concept_mapping_saved items_count=8 project='Concept Mapping Demo'


2026-04-16 00:27:00 [info     ] project.concepts_mapped        auto_rate=0.0 project='Concept Mapping Demo' total=8


Source file concept mapping:
  Total unique codes: 8
  Auto-mapped:        0
  Needs review:       0
  Coverage:           0.0%


/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/engines/polars_engine.py:131: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


## Best Practices for Concept Mapping

### Coverage Targets

For a production OMOP CDM transformation, aim for the following coverage targets:

| Domain | Target Coverage | Notes |
|---|---|---|
| Conditions (diagnoses) | >= 95% | ICD-10 to SNOMED mappings are well-established |
| Medications | >= 90% | NDC-to-RxNorm coverage depends on formulary recency |
| Lab tests | >= 85% | Local lab codes often need manual mapping |
| Procedures | >= 90% | CPT/HCPCS to SNOMED mappings are reliable |

### Review Workflow Recommendations

1. **Start with high-volume codes** -- Focus review effort on codes that appear most frequently
   in the source data. A single high-frequency code impacts more patient records.

2. **Use domain expertise** -- Have clinicians review mappings in their specialty area.
   A cardiologist should review cardiac condition mappings, etc.

3. **Document overrides** -- When you override a mapping, record why the automated match
   was not suitable. This helps improve future mapping accuracy.

4. **Validate with data quality checks** -- After mapping, run OMOP data quality checks
   (e.g., OHDSI Data Quality Dashboard) to catch systematic mapping errors.

5. **Iterate** -- Concept mapping is rarely a one-pass process. Expect to refine mappings
   as you discover edge cases during validation.

### ConceptMappingMethod Reference

| Method | Description |
|---|---|
| `AUTO` | Auto-accepted by the system (high confidence) |
| `REVIEW` | Flagged for human review (medium confidence) |
| `MANUAL` | Requires manual mapping (low confidence or no candidates) |
| `OVERRIDE` | Manually specified by a reviewer |
| `UNMAPPED` | Explicitly marked as having no suitable standard concept |

## Importing Existing Mapping Tables

If you already have a concept mapping table from a previous migration or manual curation, you can import it directly into your project.

In [20]:
import pathlib
# These files are produced by the export step above (if that cell ran successfully)
for fname in ["my_hospital_mappings.csv", "curated_mappings.csv"]:
    if pathlib.Path(fname).exists():
        import polars as pl
        print(pl.read_csv(fname))
    else:
        print(f"{fname} not found — run the export cell first")


my_hospital_mappings.csv not found — run the export cell first
curated_mappings.csv not found — run the export cell first


In [21]:
import pathlib
# These files are produced by the export step above (if that cell ran successfully)
for fname in ["my_hospital_mappings.csv", "curated_mappings.csv"]:
    if pathlib.Path(fname).exists():
        import polars as pl
        print(pl.read_csv(fname))
    else:
        print(f"{fname} not found — run the export cell first")


my_hospital_mappings.csv not found — run the export cell first
curated_mappings.csv not found — run the export cell first


In [22]:
# Import from a list of dicts (programmatic construction)
concept_map = project.import_concept_mapping(records=[
    {"source_code": "E11.9", "source_description": "Type 2 DM", "target_concept_id": 201826, "target_concept_name": "Type 2 diabetes mellitus", "target_vocabulary_id": "SNOMED", "confidence": 0.98},
    {"source_code": "I10", "source_description": "HTN", "target_concept_id": 320128, "target_concept_name": "Essential hypertension", "target_vocabulary_id": "SNOMED", "confidence": 0.95},
])
print(concept_map.summary())

2026-04-16 00:27:00 [info     ] local_storage.concept_mapping_saved items_count=2 project='Concept Mapping Demo'


2026-04-16 00:27:00 [info     ] project.concept_mapping_imported items=2 project='Concept Mapping Demo'


{'total': 2, 'auto_mapped': 0, 'needs_review': 2, 'manual_required': 0, 'auto_rate': 0.0, 'coverage': 100.0}


## Exporting Mapping Tables

Export your mapping results for clinical SME review or database loading.

In [23]:
# Export to CSV for clinical SME review
project.export_concept_mapping("mappings_for_review.csv")

# Export to JSON (includes full candidate lists and provenance)
project.export_concept_mapping("mappings_full.json")

# Export in OMOP source_to_concept_map format for database loading
project.export_concept_mapping("source_to_concept_map.csv", omop_format=True)

2026-04-16 00:27:00 [info     ] project.concept_mapping_exported omop_format=False path=mappings_for_review.csv project='Concept Mapping Demo'


2026-04-16 00:27:00 [info     ] project.concept_mapping_exported omop_format=False path=mappings_full.json project='Concept Mapping Demo'


2026-04-16 00:27:00 [info     ] project.concept_mapping_exported omop_format=True path=source_to_concept_map.csv project='Concept Mapping Demo'


'source_to_concept_map.csv'

In [24]:
# Work with mappings as a DataFrame
df = concept_map.to_dataframe()
print(df.head())

# Filter to only items needing review
review_items = concept_map.needs_review()
print(f"\n{len(review_items)} items need clinical review")

shape: (2, 10)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬────────┐
│ source_cod ┆ source_des ┆ source_co ┆ source_co ┆ … ┆ target_vo ┆ target_do ┆ confidenc ┆ method │
│ e          ┆ cription   ┆ lumn      ┆ unt       ┆   ┆ cabulary_ ┆ main_id   ┆ e         ┆ ---    │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆   ┆ id        ┆ ---       ┆ ---       ┆ str    │
│ str        ┆ str        ┆ null      ┆ i64       ┆   ┆ ---       ┆ null      ┆ f64       ┆        │
│            ┆            ┆           ┆           ┆   ┆ str       ┆           ┆           ┆        │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪════════╡
│ E11.9      ┆ Type 2 DM  ┆ null      ┆ 1         ┆ … ┆ SNOMED    ┆ null      ┆ 0.98      ┆ review │
│ I10        ┆ HTN        ┆ null      ┆ 1         ┆ … ┆ SNOMED    ┆ null      ┆ 0.95      ┆ review │
└────────────┴────────────┴───────────┴───────────┴───┴───────────┴─────────